# Task 4: Data Preprocessing Strategy: Adult (Census Income) Dataset

**Dataset:** Kaggle `adult.csv` (https://www.kaggle.com/datasets/uciml/adult-census-income), originally UCI (https://archive.ics.uci.edu/dataset/2/adult)  
**Tools:** Python (pandas, NumPy, scikit-learn, imbalanced-learn, Matplotlib, Seaborn). No spreadsheet software was used.  

> Colab already has `imbalanced-learn`. Locally run `pip install imbalanced-learn` first.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')
RS = 42

## 1. Data cleaning
Order matters: fix labels and placeholders, remove duplicates, drop redundant columns, and only then split.

In [ ]:
df = pd.read_csv('adult.csv')
print('Raw shape:', df.shape)

df = df.replace('?', np.nan)            # hidden missing values -> real NaN
before = len(df)
df = df.drop_duplicates()               # duplicate removal BEFORE the split (prevents leakage)
print('Duplicates removed:', before - len(df), '| new shape:', df.shape)

df = df.drop(columns=['education'])     # redundant with education.num (Cramer's V = 1.00)
df['income'] = (df['income'] == '>50K').astype(int)
print(df['income'].value_counts(normalize=True).round(4))

## 2. Train / test split (stratified)
The test set is set aside now; every fitted step below learns from the training set only.

In [ ]:
X = df.drop(columns='income')
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Positive rate  train:', round(y_train.mean(), 4), ' test:', round(y_test.mean(), 4))

## 3. Missing-value strategy

In [ ]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print('Missing values in training set:')
X_train.isna().sum()[lambda s: s > 0]

In [ ]:
# Is missingness informative? Income rate for rows with vs without a reported workclass
miss = X_train['workclass'].isna()
print('Positive rate when workclass MISSING :', round(y_train[miss].mean(), 3))
print('Positive rate when workclass PRESENT :', round(y_train[~miss].mean(), 3))

## 4. Transformation: skewness before and after log1p

In [ ]:
skew_cols = ['fnlwgt', 'capital.gain', 'capital.loss']
skew_tbl = pd.DataFrame({
    'skew_before': X_train[skew_cols].skew(),
    'skew_after_log1p': np.log1p(X_train[skew_cols]).skew(),
}).round(2)
skew_tbl

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for j, c in enumerate(skew_cols):
    sns.histplot(X_train[c], bins=40, ax=axes[0, j], color='#C44E52'); axes[0, j].set_title(f'{c} (before)')
    sns.histplot(np.log1p(X_train[c]), bins=40, ax=axes[1, j], color='#4C72B0'); axes[1, j].set_title(f'{c} (after log1p)')
plt.tight_layout(); plt.savefig('figures/fig_log_transform.png', dpi=150); plt.show()

## 5. Preprocessing pipeline (imputation, log, scaling, encoding)

In [ ]:
num_log = ['capital.gain', 'capital.loss']
num_std = ['age', 'education.num', 'hours.per.week']      # fnlwgt dropped (see section 6)

def build_preprocessor(use_log=True):
    log_step = Pipeline([('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                         ('scale', StandardScaler())]) if use_log else StandardScaler()
    cat_step = Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
                         ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                  min_frequency=0.01, sparse_output=False))])
    return ColumnTransformer([('cap', log_step, num_log),
                              ('num', StandardScaler(), num_std),
                              ('cat', cat_step, cat_cols)])

prep = build_preprocessor(use_log=False)   # main pipeline: scaling only (log tested in section 9)
Xtr = prep.fit_transform(X_train)          # fit on TRAIN only
Xte = prep.transform(X_test)               # transform TEST with train statistics
feature_names = prep.get_feature_names_out()
print('Features before encoding:', X_train.shape[1], '| after encoding:', Xtr.shape[1])
print('Any NaN left in train?', np.isnan(Xtr).any())
full_ohe = pd.get_dummies(X_train.fillna('Unknown'), columns=cat_cols).shape[1]
print('Columns if every category is one-hot encoded (no grouping):', full_ohe)

## 6. Feature selection (mutual information)

In [ ]:
mi = pd.Series(mutual_info_classif(Xtr, y_train, random_state=RS), index=feature_names).sort_values(ascending=False)
mi.head(10).round(3)

In [ ]:
# Raw numeric relevance, including fnlwgt (justifies dropping it)
raw_num = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
pd.Series(mutual_info_classif(X_train[raw_num], y_train, random_state=RS), index=raw_num).sort_values(ascending=False).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
mi.head(15).sort_values().plot.barh(ax=ax, color='#55A868')
ax.set_title('Top 15 features by mutual information with income')
plt.tight_layout(); plt.savefig('figures/fig_mi.png', dpi=150); plt.show()

## 7. Dimensionality reduction (PCA)

In [ ]:
pca = PCA(random_state=RS).fit(Xtr)
cum = np.cumsum(pca.explained_variance_ratio_)
print('Components for 90% variance:', int(np.argmax(cum >= 0.90)) + 1)
print('Components for 95% variance:', int(np.argmax(cum >= 0.95)) + 1)
print('Total encoded features     :', Xtr.shape[1])

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cum) + 1), cum, marker='.')
plt.axhline(0.95, color='r', ls='--', label='95% variance')
plt.xlabel('Number of components'); plt.ylabel('Cumulative explained variance'); plt.legend()
plt.title('PCA explained variance'); plt.tight_layout(); plt.savefig('figures/fig_pca.png', dpi=150); plt.show()

## 8. Evaluation helper (metrics suited to imbalance)

In [ ]:
def evaluate(name, model, Xa, ya, Xb, yb):
    model.fit(Xa, ya)
    pred = model.predict(Xb); prob = model.predict_proba(Xb)[:, 1]
    return {'setup': name,
            'precision': precision_score(yb, pred), 'recall': recall_score(yb, pred),
            'f1': f1_score(yb, pred), 'roc_auc': roc_auc_score(yb, prob),
            'pr_auc': average_precision_score(yb, prob)}

def lr(**kw): return LogisticRegression(max_iter=2000, random_state=RS, **kw)
def rf(**kw): return RandomForestClassifier(n_estimators=200, random_state=RS, n_jobs=-1, **kw)

## 9. Do the log transform, feature selection and PCA help? (Logistic regression)

In [ ]:
res = []
# (a) scaling only (main pipeline)
res.append(evaluate('LR, scaling only (main)', lr(), Xtr, y_train, Xte, y_test))
# (b) log1p on capital.gain / capital.loss, then scaling
p1 = build_preprocessor(use_log=True)
res.append(evaluate('LR, log1p + scaling', lr(), p1.fit_transform(X_train), y_train, p1.transform(X_test), y_test))
# (c) top-30 features by mutual information
top30 = [list(feature_names).index(f) for f in mi.head(30).index]
res.append(evaluate('LR, top-30 MI features', lr(), Xtr[:, top30], y_train, Xte[:, top30], y_test))
# (d) PCA keeping 95% variance
pca95 = PCA(n_components=0.95, random_state=RS).fit(Xtr)
res.append(evaluate('LR, PCA 95% variance', lr(), pca95.transform(Xtr), y_train, pca95.transform(Xte), y_test))
pd.DataFrame(res).set_index('setup').round(3)

## 10. Class balancing (resample the TRAINING set only)

In [ ]:
print('Train class counts before:', dict(y_train.value_counts()))
Xs, ys = SMOTE(random_state=RS).fit_resample(Xtr, y_train)
Xo, yo = RandomOverSampler(random_state=RS).fit_resample(Xtr, y_train)
Xu, yu = RandomUnderSampler(random_state=RS).fit_resample(Xtr, y_train)
for n, yy in [('SMOTE', ys), ('Random over', yo), ('Random under', yu)]:
    print(n, '->', dict(pd.Series(yy).value_counts()))

In [ ]:
bal = []
for mname, mk in [('Logistic regression', lr), ('Random forest', rf)]:
    bal.append(evaluate(f'{mname}: none', mk(), Xtr, y_train, Xte, y_test))
    bal.append(evaluate(f'{mname}: class_weight', mk(class_weight='balanced'), Xtr, y_train, Xte, y_test))
    bal.append(evaluate(f'{mname}: random over-sampling', mk(), Xo, yo, Xte, y_test))
    bal.append(evaluate(f'{mname}: random under-sampling', mk(), Xu, yu, Xte, y_test))
    bal.append(evaluate(f'{mname}: SMOTE', mk(), Xs, ys, Xte, y_test))
bal_tbl = pd.DataFrame(bal).set_index('setup').round(3)
bal_tbl

In [ ]:
ax = bal_tbl[['precision', 'recall', 'f1']].plot.barh(figsize=(8, 6))
ax.set_title('Effect of balancing strategy on test-set metrics')
plt.tight_layout(); plt.savefig('figures/fig_balancing.png', dpi=150); plt.show()

## 11. Final pipeline

In [ ]:
final = Pipeline([('prep', build_preprocessor(use_log=False)),
                  ('model', lr(class_weight='balanced'))])
final.fit(X_train, y_train)
pred = final.predict(X_test); prob = final.predict_proba(X_test)[:, 1]
print('Final pipeline steps:', [s for s, _ in final.steps])
print('precision', round(precision_score(y_test, pred), 3), '| recall', round(recall_score(y_test, pred), 3),
      '| f1', round(f1_score(y_test, pred), 3), '| roc_auc', round(roc_auc_score(y_test, prob), 3),
      '| pr_auc', round(average_precision_score(y_test, prob), 3))